# 06. Pipeline de Transformação Unificado - Anti Data Leakage

## 🎯 Objetivo

Implementar um **pipeline completo de transformação de dados** que garante **ZERO data leakage**
através da aplicação estrita da regra: **Fit APENAS no treino, Transform em treino e OOT**.

## 📚 Transformações Aplicadas

### 1. **Datetime Feature Extraction**
- Componentes temporais (year, month, day, hour, etc.)
- Features cíclicas (sin/cos para capturar periodicidade)
- Features de negócio (is_weekend, is_business_hours, is_night)

### 2. **Categorical Encoding**
- **One-Hot Encoding**: Variáveis de baixa cardinalidade (≤ 10 categorias)
- **Frequency Encoding**: Variáveis de alta cardinalidade (> 10 categorias)

### 3. **Numerical Transformations**
- **Imputação**: Mediana aprendida no treino
- **Normalização**: StandardScaler ajustado no treino

## ⚠️ Garantia Anti-Leakage

✅ **ColumnTransformer sklearn**: Organiza transformações por tipo de variável  
✅ **Pipeline sklearn**: Sequencia operações de forma reprodutível  
✅ **Fit APENAS no X_train**: Parâmetros aprendidos somente do treino  
✅ **Transform em X_train e X_oot**: Usa parâmetros aprendidos do treino

---

## 1. Setup - Importações e Configuração

In [ ]:
# Configurar path para importar módulos do projeto
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Importar configurações do projeto
from source.config import PROJ_ROOT, get_data_path, get_model_path, ensure_directories

# Garantir que os diretórios existam
ensure_directories()

print(f"✅ Projeto Root: {PROJ_ROOT}")
print(f"✅ Configurações carregadas de source/config.py")

In [ ]:
# Importações padrão
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import json
import joblib

# Scikit-Learn
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Importar módulo de preprocessing customizado
from source.preprocessing import (
    DateTimeFeatureExtractor,
    FrequencyEncoder,
    OneHotEncoderSafe,
    ImputerWithStrategy,
    build_safe_preprocessing_pipeline,
    apply_preprocessing_pipeline
)

# Configurações
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)

print("✅ Bibliotecas importadas com sucesso!")

## 2. Carregamento dos Dados com Features

Carregamos os dados **COM features de engenharia** gerados no Notebook 05.

In [ ]:
print("="*80)
print("CARREGAMENTO DOS DADOS")
print("="*80)

# Carregar datasets COM features geradas no Notebook 05
df_treino = pd.read_csv(get_data_path('df_treino_with_features.csv', 'processed'))
df_oot = pd.read_csv(get_data_path('df_oot_with_features.csv', 'processed'))

# Converter Timestamp para datetime
df_treino['Timestamp'] = pd.to_datetime(df_treino['Timestamp'])
df_oot['Timestamp'] = pd.to_datetime(df_oot['Timestamp'])

print(f"\n📊 Dataset de Treino:")
print(f"   Shape: {df_treino.shape}")
print(f"   Período: {df_treino['Timestamp'].min()} até {df_treino['Timestamp'].max()}")

print(f"\n📊 Dataset de OOT:")
print(f"   Shape: {df_oot.shape}")
print(f"   Período: {df_oot['Timestamp'].min()} até {df_oot['Timestamp'].max()}")

print(f"\n✅ Dados carregados com sucesso!")

## 3. Separação de Features e Target

Separamos X (features) e y (target) para treino e OOT.

In [ ]:
# Definir target
target_col = 'Is Laundering'

# Separar X e y
X_train = df_treino.drop(columns=[target_col])
y_train = df_treino[target_col]

X_oot = df_oot.drop(columns=[target_col])
y_oot = df_oot[target_col]

print("="*80)
print("SEPARAÇÃO DE FEATURES E TARGET")
print("="*80)

print(f"\n📊 Treino:")
print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   Taxa de lavagem: {y_train.mean()*100:.2f}%")

print(f"\n📊 OOT:")
print(f"   X_oot shape: {X_oot.shape}")
print(f"   y_oot shape: {y_oot.shape}")
print(f"   Taxa de lavagem: {y_oot.mean()*100:.2f}%")

## 4. Construção do Pipeline Seguro

Usamos a função `build_safe_preprocessing_pipeline` de `source.preprocessing` para construir
um pipeline que garante fit apenas no treino.

In [ ]:
# Construir pipeline
pipeline, feature_names = build_safe_preprocessing_pipeline(
    df_treino=X_train,
    target_col=target_col,
    datetime_cols=['Timestamp'],  # Especificar Timestamp para extração
    categorical_cols=None,  # Auto-detectar
    numeric_cols=None  # Auto-detectar
)

print("\n="*80)
print("PIPELINE CONSTRUÍDO")
print("="*80)

print(f"\n📦 Pipeline sklearn.pipeline.Pipeline:")
print(pipeline)

## 5. Aplicação do Pipeline - Fit e Transform

⚠️ **CRÍTICO**: Fit APENAS no X_train, Transform em X_train e X_oot.

In [ ]:
# Aplicar pipeline
X_train_transformed, X_oot_transformed = apply_preprocessing_pipeline(
    pipeline=pipeline,
    X_train=X_train,
    X_oot=X_oot,
    y_train=y_train,  # Necessário se houver Target Encoding
    save_path=get_model_path('preprocessing_pipeline.pkl')
)

print("\n="*80)
print("TRANSFORMAÇÃO CONCLUÍDA")
print("="*80)

print(f"\n📊 Shapes transformados:")
print(f"   X_train: {X_train.shape} → {X_train_transformed.shape}")
print(f"   X_oot: {X_oot.shape} → {X_oot_transformed.shape}")

print(f"\n📝 Colunas finais: {X_train_transformed.shape[1]}")
print(f"\n✅ Pipeline salvo em: {get_model_path('preprocessing_pipeline.pkl')}")

## 6. Análise dos Dados Transformados

Verificar distribuições e estatísticas após transformação.

In [ ]:
# Estatísticas descritivas
print("="*80)
print("ESTATÍSTICAS DESCRITIVAS - DADOS TRANSFORMADOS (Treino)")
print("="*80)

print(f"\n📊 Primeiras colunas:")
print(X_train_transformed.describe().iloc[:, :10].T)

print(f"\n📊 Últimas colunas:")
print(X_train_transformed.describe().iloc[:, -10:].T)

In [ ]:
# Verificar valores ausentes
print("="*80)
print("VALORES AUSENTES APÓS TRANSFORMAÇÃO")
print("="*80)

nan_train = X_train_transformed.isnull().sum().sum()
nan_oot = X_oot_transformed.isnull().sum().sum()

print(f"\n✓ Treino: {nan_train} valores ausentes")
print(f"✓ OOT: {nan_oot} valores ausentes")

if nan_train == 0 and nan_oot == 0:
    print("\n✅ Nenhum valor ausente! Pipeline de imputação funcionou corretamente.")
else:
    print("\n⚠️  Ainda há valores ausentes. Verificar pipeline de imputação.")

In [ ]:
# Verificar normalização (média ~0, std ~1)
print("="*80)
print("VALIDAÇÃO DA NORMALIZAÇÃO (StandardScaler)")
print("="*80)

# Amostra de colunas numéricas
numeric_sample = X_train_transformed.select_dtypes(include=[np.number]).columns[:20]

means = X_train_transformed[numeric_sample].mean()
stds = X_train_transformed[numeric_sample].std()

print(f"\n📊 Médias (devem estar próximas de 0):")
print(f"   Min: {means.min():.4f}")
print(f"   Max: {means.max():.4f}")
print(f"   Média das médias: {means.mean():.4f}")

print(f"\n📊 Desvios padrão (devem estar próximos de 1):")
print(f"   Min: {stds.min():.4f}")
print(f"   Max: {stds.max():.4f}")
print(f"   Média dos stds: {stds.mean():.4f}")

if abs(means.mean()) < 0.1 and abs(stds.mean() - 1.0) < 0.2:
    print("\n✅ Normalização bem-sucedida!")
else:
    print("\n⚠️  Normalização pode ter problemas. Revisar pipeline.")

## 7. Validação Anti-Leakage

Verificar que não há vazamento de informação do OOT para o treino.

In [ ]:
print("="*80)
print("VALIDAÇÃO ANTI-LEAKAGE")
print("="*80)

# Verificação 1: Médias e stds do OOT devem ser DIFERENTES do treino
# (porque foram normalizados com parâmetros do treino, não do OOT)

sample_cols = X_train_transformed.select_dtypes(include=[np.number]).columns[:10]

train_means = X_train_transformed[sample_cols].mean()
oot_means = X_oot_transformed[sample_cols].mean()

train_stds = X_train_transformed[sample_cols].std()
oot_stds = X_oot_transformed[sample_cols].std()

print("\n✓ Verificação 1: Estatísticas do OOT diferem do treino?")
print(f"\n  Diferença média das médias: {abs(train_means - oot_means).mean():.4f}")
print(f"  Diferença média dos stds: {abs(train_stds - oot_stds).mean():.4f}")

if abs(train_means - oot_means).mean() > 0.01:
    print("\n  ✅ OOT tem estatísticas diferentes → Pipeline aplicado corretamente!")
else:
    print("\n  ⚠️  OOT tem estatísticas muito similares → Possível leakage!")

# Verificação 2: Re-aplicar transform deve dar mesmo resultado
print("\n✓ Verificação 2: Idempotência (re-transform = mesmo resultado)")

X_train_check = pipeline.transform(X_train.head(100))
X_train_original = X_train_transformed.head(100)

if isinstance(X_train_check, np.ndarray):
    X_train_check = pd.DataFrame(X_train_check, columns=X_train_transformed.columns)

diff = abs(X_train_check.values - X_train_original.values).max()

print(f"  Diferença máxima: {diff:.10f}")

if diff < 1e-10:
    print("\n  ✅ Transform é idempotente!")
else:
    print(f"\n  ⚠️  Transform não é idempotente! Diferença: {diff}")

print("\n✅ Validações anti-leakage concluídas!")

## 8. Salvamento dos Dados Transformados

Salvamos X_train, X_oot, y_train, y_oot para uso no treinamento de modelos.

In [ ]:
print("="*80)
print("SALVAMENTO DOS DADOS TRANSFORMADOS")
print("="*80)

# Definir caminhos
path_X_train = get_data_path('X_train.csv', 'processed')
path_X_oot = get_data_path('X_oot.csv', 'processed')
path_y_train = get_data_path('y_train.csv', 'processed')
path_y_oot = get_data_path('y_oot.csv', 'processed')

# Salvar
X_train_transformed.to_csv(path_X_train, index=False)
X_oot_transformed.to_csv(path_X_oot, index=False)
y_train.to_csv(path_y_train, index=False, header=True)
y_oot.to_csv(path_y_oot, index=False, header=True)

print(f"\n✅ Datasets transformados salvos com sucesso!")
print(f"\n📁 Arquivos:")
print(f"   - {path_X_train}")
print(f"   - {path_X_oot}")
print(f"   - {path_y_train}")
print(f"   - {path_y_oot}")

# Verificar tamanhos
size_X_train = Path(path_X_train).stat().st_size / (1024 * 1024)
size_X_oot = Path(path_X_oot).stat().st_size / (1024 * 1024)

print(f"\n📏 Tamanhos:")
print(f"   - X_train: {size_X_train:.2f} MB")
print(f"   - X_oot: {size_X_oot:.2f} MB")

## 9. Teste de Carregamento do Pipeline

Verificar que o pipeline salvo pode ser recarregado e usado corretamente.

In [ ]:
print("="*80)
print("TESTE DE CARREGAMENTO DO PIPELINE")
print("="*80)

# Carregar pipeline salvo
pipeline_loaded = joblib.load(get_model_path('preprocessing_pipeline.pkl'))

print(f"\n✅ Pipeline carregado com sucesso!")

# Testar transform em amostra
sample = X_train.head(10)
transformed_sample = pipeline_loaded.transform(sample)

print(f"\n✓ Teste de transform:")
print(f"   Input shape: {sample.shape}")
print(f"   Output shape: {transformed_sample.shape}")

print("\n✅ Pipeline salvo está funcional!")

## 10. Sumário e Próximos Passos

### ✅ Realizações deste Notebook

1. ✅ Construção de **Pipeline sklearn** unificado
2. ✅ Aplicação de **Fit APENAS no treino**
3. ✅ Transformação consistente em **treino e OOT**
4. ✅ **Datetime extraction** (componentes + cíclicas + negócio)
5. ✅ **Categorical encoding** (One-Hot + Frequency)
6. ✅ **Imputação** (mediana aprendida no treino)
7. ✅ **Normalização** (StandardScaler ajustado no treino)
8. ✅ Validação **anti-leakage**
9. ✅ Persistência do pipeline

### 📊 Estatísticas Finais

- **Features de entrada**: {X_train.shape[1]}
- **Features de saída**: {X_train_transformed.shape[1]}
- **Samples treino**: {X_train_transformed.shape[0]:,}
- **Samples OOT**: {X_oot_transformed.shape[0]:,}

### 🔄 Próximos Passos

**Notebook 08**: Treinamento de Modelos com:
- TimeSeriesSplit para validação temporal
- Balanceamento correto (SMOTE dentro dos folds)
- Threshold financeiro otimizado (Matriz de Custo)

---

**Data**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Status**: ✅ Pipeline de Transformação Concluído